# ATML PA3 — Task 3: Test-Time Compute: Sampling, Selection, and Adaptive Inference
## Complete Reproducible Google Colab Runbook

**Course**: Advanced Topics in Machine Learning (ATML), Fall 2026  
**Target Runtime**: Google Colab with NVIDIA T4 GPU (16 GiB VRAM) & Persistent Google Drive  

This runbook executes all components of **Task 3**:
1. **Clones / pulls** the latest code from your GitHub repository (`https://github.com/Saad-0000/atml-pa3-FineTuning.git`).
2. **Connects** persistent storage via Google Drive symlinks (`checkpoints`, `artifacts`, `results`, `logs`).
3. **Offline Rollouts Analysis**: 1,000 MATH and 1,000 MathVista questions (32 rollouts per question = 64,000 total).
4. **Live Model Generation**: Single-long, parallel, and sequential refinement on first 50 MATH held-out questions using `Qwen/Qwen2.5-3B-Instruct` (FP16, batch size 1).
5. **Adaptive Inference**: Grid-search calibration on development set (200 questions) and frozen evaluation on held-out set (800 questions).
6. **Compiles** all 18 Tables (`T00`–`T17`), 16 Figures (`P01`–`P16`, PNG & PDF), and audits evidence completeness.

## 1. Mount Google Drive, Clone/Pull Repository & Setup Persistent Storage
Mount Google Drive for persistent storage, check GPU availability, pull the latest code from GitHub, and set up symlinks so all progress is crash-safe.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

# 1. Clone or pull the latest code from your GitHub repository
%cd /content
REPO_URL = 'https://github.com/Saad-0000/atml-pa3-FineTuning.git'
![ -d repo ] && (cd repo && git pull origin master) || git clone {REPO_URL} repo
%cd /content/repo

# 2. Setup persistent Google Drive directories and symlinks for crash-safety
PERSISTENT_ROOT = '/content/drive/MyDrive/ATML_PA3_Task3'
!mkdir -p {PERSISTENT_ROOT}/checkpoints
!mkdir -p {PERSISTENT_ROOT}/artifacts
!mkdir -p {PERSISTENT_ROOT}/results
!mkdir -p {PERSISTENT_ROOT}/logs

# Ensure persistent storage is preserved across Colab restarts
!mkdir -p task3/artifacts task3/checkpoints task3/logs task3/results
!cp -r -n task3/results/* {PERSISTENT_ROOT}/results/ 2>/dev/null || true
!rm -rf task3/artifacts task3/checkpoints task3/logs task3/results
!ln -s {PERSISTENT_ROOT}/artifacts task3/artifacts
!ln -s {PERSISTENT_ROOT}/checkpoints task3/checkpoints
!ln -s {PERSISTENT_ROOT}/logs task3/logs
!ln -s {PERSISTENT_ROOT}/results task3/results

import os
os.environ['PYTHONPATH'] = f'/content/repo/task3/src:{os.environ.get("PYTHONPATH", "")}'
print(f"Current working directory: {os.getcwd()}")
print(f"Persistent Drive root: {PERSISTENT_ROOT}")

# Path to the dataset provided for Task 3 on Google Drive
DATA_DIR = "/content/drive/MyDrive/ATML-PA3-Data"
print(f"Dataset Directory: {DATA_DIR}")


## 2. Install Task 3 Dependencies
Installs pinned dependencies from `task3/requirements.txt` and `math-verify`.

In [ ]:
!pip install -q -r task3/requirements.txt
!pip install -q math-verify


## 3. Run Preflight Check & Environment Validation
Verifies CUDA GPU, torch installation, parquet data directory integrity, and compute prerequisites.

In [ ]:
!python task3/scripts/preflight.py --data_dir {DATA_DIR}


## 4. Run Automated Unit Tests (21 Tests)
Executes the full unit test suite covering split determinism, scoring/normalization, no-leakage verification, signal fusion, live generation parsing, and adaptive controllers.

In [ ]:
!python -m unittest discover -s task3/tests


## (Option A) One-Click End-to-End Orchestrator (Crash-Resilient)
If you prefer to run the entire Task 3 pipeline with automatic checkpointing and crash recovery in a single command, execute the cell below.
- If Colab disconnects, simply re-run this cell and it will automatically detect existing checkpoint artifacts and resume from where it left off without re-running completed stages.
- If you prefer to execute step-by-step instead, skip this cell and run Steps 5–15 individually below.

In [ ]:
!python task3/scripts/resume.py --data_dir {DATA_DIR}


---
## (Option B) Step-by-Step Execution
The following cells run each stage modularly.

### Step 5: Data Inspection & Schema Verification (Table T00)

In [ ]:
!python task3/scripts/inspect_data.py --data_dir {DATA_DIR}


### Step 6: Build Deterministic Splits (Table T01)
Partitions 1,000 questions each for MATH and MathVista into 200 development and 800 held-out questions using seed `6304`.

In [ ]:
!python task3/scripts/build_splits.py --data_dir {DATA_DIR}


### Step 7: Rollout Normalization & Scoring (Table T02)
Extracts answers with balanced-brace parsing, applies MATH/MathVista normalizers and SymPy verification, and scores all 64,000 rollouts.

In [ ]:
!python task3/scripts/prepare_data.py --data_dir {DATA_DIR}


### Step 8: Offline Sampling & Selection Experiments A & B (Tables T03, T04, T05)
Evaluates FirstAns, Self-Consistency, Highest-Score (mean logp, certainty, vision, VisConf), and Rank-Weighted Consensus across budgets N in {1, 4, 8, 16, 32} with 10 random permutations.

In [ ]:
!python task3/scripts/run_offline_sampling.py


### Step 9: Offline Signal & Fusion Analysis Experiment C (Tables T06, T07, T08)
Analyzes quartile accuracy, tokenwise correlation between confidence and visual weight, and compares tokenwise vs trajectory-level signal fusion.

In [ ]:
!python task3/scripts/run_signal_analysis.py


### Step 10: Live Generation Experiment (Tables T09, T10, T11)
Loads `Qwen/Qwen2.5-3B-Instruct` in FP16 (no quantization, batch size 1) and executes single-long, parallel, and sequential refinement on the first 50 MATH held-out questions (equal budget of 3,072 tokens).

In [ ]:
!python task3/scripts/run_live_generation.py --data_dir {DATA_DIR}


### Step 11: Adaptive Policy Tuning on Development Split (Table T12)
Performs grid search over agreement thresholds tau in {0.5..0.9} and quality weights gamma in {0.0..1.0} on the 200 dev questions, selecting policies within 1 pp of fixed-32 SC with minimal token cost.

In [ ]:
!python task3/scripts/run_adaptive_tuning.py


### Step 12: Adaptive Policy Evaluation on Held-out Split (Tables T13, T14)
Applies the frozen selected policies to the 800 held-out questions and measures accuracy, cost savings, stopping distributions, and premature stop recovery.

In [ ]:
!python task3/scripts/run_adaptive_test.py


### Step 13: Qualitative Case Index (Table T15)
Extracts four required qualitative case studies: selection disagreement, signal trajectory, refinement gain, and adaptive premature stop.

In [ ]:
!python task3/scripts/build_qualitative_cases.py


### Step 14: Compile All Tables (T00 – T17) and Render All Plots (P01 – P16)
Builds all required CSV/JSON tables and publication-quality plots (in both PNG and vector PDF).

In [ ]:
!python task3/scripts/make_tables.py
!python task3/scripts/make_plots.py


### Step 15: Run Final Deliverables & Evidence Audit
Audits evidence completeness and prints the human-readable audit report.

In [ ]:
!python task3/scripts/audit_evidence.py
with open('task3/results/EVIDENCE_AUDIT.md', 'r') as f:
    print(f.read())
